# E03 — Dynamic workload

| | |
|---|---|
| **Question** | Vessim treats demand as a power signal. How much effort does it take to add jobs, queues and temporal shifting, and what does it give compared with the edge study's scheduler? |
| **Hypothesis** | H1 (Vessim has no workload model) |
| **Branch** | `exp/e03-dynamic-workload` |
| **Site** | Edge study S1.2, 2025-04-01, one day: job trace `aggregated_jobs_60_OL_20` (7 279 jobs, 20 % delay-tolerant), 614 000-CPU pool, building and PV from the edge run, day-ahead price; grid only (no battery) |

**Variants**

| Variant | How it is built in Vessim |
|---|---|
| Static IT trace (edge study output) | built-in `Consumer` with the edge run's IT and cooling power traces — Vessim's native view of demand |
| Jobs, start on arrival | custom `ClusterActor`: every job starts when it arrives |
| Jobs, cheapest price | same actor; delay-tolerant jobs start at the slot within their allowed delay with the lowest mean day-ahead price over their runtime |
| Jobs, lowest peak | same actor; delay-tolerant jobs start where the predicted site import over their runtime peaks lowest (≈ the edge study's greedy `peak_load` scheduler) |
| Jobs, hold on high import | same actor + custom `Controller`: delay-tolerant jobs are held while observed grid import exceeds a threshold, forced to start at their deadline |

The job model: jobs have CPU demand, runtime and a maximum start delay (0 for 80 % of jobs). IT power follows the edge
study's linear CPU model (`simulators/hardware.py`: 900 kW base load + 7.3 W per CPU at peak, 21.7 % idle). Vessim has no
thermal model, so cooling is a linear function of IT power fitted to the edge run (E04 looks at thermal modelling).

## 1. Setup

In [1]:
import importlib.metadata as md_
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from vessim_eval import DATA_DIR, RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.dispatch import logic_lines, price_from_reference
from vessim_eval.edge_reference import load_reference, scenario_config
from vessim_eval.scenarios import agreement
from vessim_eval.workload import (ClusterActor, CoolingProxy, ImportHoldController, ServerPool,
                                  run_static, run_workload, workload_kpis)

print("vessim", md_.version("vessim"))

vessim 0.16.0


In [2]:
# ---- Parameters -------------------------------------------------------------
SCENARIO = "s1_2_battery_pv_large"
STEP_S = 60                                   # job model step (edge scheduler: 30 s)
HOLD_THRESHOLD_KW = 6000.0                    # reactive hold: observed grid import threshold
HOLD_SWEEP_KW = [5600, 5800, 6000, 6200]
HOLD_STEPS_S = {"1 min": 60, "5 min": 300, "15 min": 900}
OUT = RESULTS_DIR / "E03"
OUT.mkdir(parents=True, exist_ok=True)

hw = scenario_config(SCENARIO)["data_centers"][0]["hardware"]
pool = ServerPool(capacity_cpu=hw["pool_settings"][0]["instance_quantity"],
                  p_peak_w_per_cpu=hw["p_peak_w_per_cpu"],
                  idle_fraction=hw["p_idle_perc_of_peak"] / 100, base_load_kw=hw["base_load"])
jobs = pd.read_csv(DATA_DIR / "e03_edge_jobs_60_OL_20.csv")
ref = load_reference("e01_edge_s1_2_15min.csv")
price = price_from_reference(ref)                                     # €/MWh
cooling = tuple(np.polyfit(ref.it_mean_kw, ref.cooling_mean_kw, 1))   # kW = a · IT + b

flex = jobs[jobs.delay_tolerant]
pd.Series({
    "Jobs (delay-tolerant)": f"{len(jobs):,} ({len(flex):,})",
    "Allowed start delay of delay-tolerant jobs": ", ".join(
        f"{m / 60:.0f} min: {n}" for m, n in flex.max_delay_s.value_counts().sort_index().items()),
    "CPU-hours (delay-tolerant share)": f"{(jobs.cpu * jobs.runtime_s).sum() / 3600:,.0f} "
        f"({(flex.cpu * flex.runtime_s).sum() / (jobs.cpu * jobs.runtime_s).sum():.0%})",
    "Server pool": f"{pool.capacity_cpu:,.0f} CPUs, IT power {pool.power_kw(0):,.0f}–"
                   f"{pool.power_kw(pool.capacity_cpu):,.0f} kW",
    "Cooling proxy": f"{cooling[0]:.3f} · IT {cooling[1]:+.1f} kW",
}, name="Value").to_frame()

,Value
Jobs (delay-tolerant),"7,279 (1,456)"
Allowed start delay of delay-tolerant jobs,"30 min: 415, 60 min: 292, 120 min: 265, 180 mi..."
CPU-hours (delay-tolerant share),"8,341,396 (20%)"
Server pool,"614,000 CPUs, IT power 1,873–5,382 kW"
Cooling proxy,0.284 · IT -113.8 kW


In [3]:
hour = pd.to_datetime(jobs.arrival_s, unit="s").dt.strftime("%H:00")
cpu_h = jobs.cpu * jobs.runtime_s / 3600
arrivals = (pd.DataFrame({"Delay-intolerant jobs": cpu_h.where(~jobs.delay_tolerant, 0),
                          "Delay-tolerant jobs": cpu_h.where(jobs.delay_tolerant, 0)})
            .groupby(hour.values).sum() / 1e3)
fig = vp.grouped_bars(arrivals.round(0), title="Arriving work per hour (job trace of the edge study)",
                      y_title="Arriving work (thousand CPU-hours)", x_title="Arrival hour")
fig.update_traces(text=None)
fig.show()

## 2. Run

In [4]:
NAMES = {"on_arrival": "Jobs, start on arrival", "cheapest_price": "Jobs, cheapest price",
         "lowest_peak": "Jobs, lowest peak", "hold": "Jobs, hold on high import"}
STATIC = "Static IT trace (edge study output)"
site = dict(building_kw=ref.building_kw, pv_kw=ref.pv_kw, price=price, cooling=cooling)

runs = {STATIC: run_static(ref.it_mean_kw, ref.cooling_mean_kw, ref.building_kw, ref.pv_kw, price)}
for rule, name in NAMES.items():
    runs[name] = run_workload(jobs, pool, rule=rule, step_s=STEP_S,
                              hold_threshold_kw=HOLD_THRESHOLD_KW, **site)
hold_sweep = {t: run_workload(jobs, pool, rule="hold", step_s=STEP_S, hold_threshold_kw=t, **site)
              for t in HOLD_SWEEP_KW}
hold_steps = {lbl: run_workload(jobs, pool, rule="hold", step_s=s, hold_threshold_kw=HOLD_THRESHOLD_KW,
                                **site) for lbl, s in HOLD_STEPS_S.items()}
for name, r in runs.items():
    print(f"{name:<38} {r['wall_s']:6.2f} s")

Static IT trace (edge study output)      0.01 s
Jobs, start on arrival                   0.64 s
Jobs, cheapest price                     5.28 s
Jobs, lowest peak                        3.63 s
Jobs, hold on high import                0.66 s


## 3. Results

### 3.1 Key numbers

In [5]:
base_it = runs[NAMES["on_arrival"]]["ts"].it_kw
kpis = pd.DataFrame({n: workload_kpis(r, base_it if r["jobs"] is not None else None)
                     for n, r in runs.items()})
kpis = kpis.reindex(workload_kpis(runs[NAMES["on_arrival"]], base_it).index)   # keep row order
kpis.to_csv(OUT / "workload_kpis.csv")
vp.format_table(kpis, 1)

,Static IT trace (edge study output),"Jobs, start on arrival","Jobs, cheapest price","Jobs, lowest peak","Jobs, hold on high import"
IT energy in the day (kWh),"90,465.4","91,046.5","91,002.5","90,420.1","91,046.5"
IT energy after midnight (kWh),—,"1,575.4","1,619.4","2,201.8","1,575.4"
Grid import energy (kWh),"110,533.7","111,279.6","111,223.1","110,475.5","111,279.6"
Energy cost (€),"10,498.6","10,633.1","10,430.0","10,487.6","10,632.5"
Cost incl. job energy after midnight (€),—,"10,817.7","10,619.7","10,745.5","10,817.0"
Peak grid import (kW),"6,252.9","6,691.1","6,336.9","6,265.2","6,536.8"
"Peak grid import, 15-min mean (kW)","6,252.9","6,620.1","6,252.7","6,234.7","6,435.3"
Peak IT power (kW),"4,940.6","5,289.9","5,013.9","4,958.1","5,169.7"
Delay-tolerant jobs shifted,—,0.0,934.0,"1,455.0",145.0
"Mean start delay, delay-tolerant (min)",—,0.0,60.2,96.1,2.6


Peak grid import is shown on 1-min values and on 15-min means; only the 15-min mean is comparable with the static
trace, which has a 15-min resolution. *IT energy after midnight* is the job energy of jobs still running at the end of
the day: all variants run the same jobs, but shifting moves some of their energy past midnight. *Cost incl. job energy
after midnight* values that energy (plus its cooling) at the day's mean price and is the fair cost comparison.

### 3.2 Job model vs the edge study

In [6]:
edge_it = ref.it_mean_kw
cmp = pd.DataFrame({n: agreement(edge_it, runs[n]["ts"].it_kw.resample("15min").mean(), 900)
                    for n in (NAMES["on_arrival"], NAMES["lowest_peak"])}).T
cmp["Daily IT energy, edge study (kWh)"] = edge_it.sum() * 0.25
cmp["Peak IT power, 15-min mean (kW)"] = [runs[n]["ts"].it_kw.resample("15min").mean().max()
                                          for n in cmp.index]
cmp["Peak IT power, edge study (kW)"] = edge_it.max()
vp.format_table(cmp, 1)

,Maximum absolute difference,Mean absolute difference,Daily energy difference (kWh),"Daily IT energy, edge study (kWh)","Peak IT power, 15-min mean (kW)","Peak IT power, edge study (kW)"
"Jobs, start on arrival",349.0,95.3,581.0,"90,465.4","5,234.6","4,940.6"
"Jobs, lowest peak",213.1,10.9,-45.3,"90,465.4","4,934.3","4,940.6"


In [7]:
q = lambda n, col: runs[n]["ts"][col].resample("15min").mean()
fig = vp.stacked_panels([
    ("IT power, 15-min mean (kW)", {n: q(n, "it_kw") for n in runs}),
    ("Grid import, 15-min mean (kW)", {n: q(n, "grid_kw") for n in runs}),
], title="IT power and grid import per workload variant", row_height=320)
vp.save(fig, OUT / "it_and_grid", png=True)
fig.show()

### 3.3 Queue and start delays

In [8]:
fig = vp.line_comparison(
    {n: runs[n]["ts"].waiting_jobs for n in NAMES.values() if n != NAMES["on_arrival"]},
    title="Jobs waiting to start (arrived, start time not yet reached)",
    y_title="Waiting jobs (count)")
vp.save(fig, OUT / "waiting_jobs", png=True)
fig.show()

In [9]:
bins = [-1, 0, 30 * 60, 60 * 60, 120 * 60, 180 * 60]
labels = ["no delay", "≤ 30 min", "30–60 min", "1–2 h", "2–3 h"]
delays = pd.DataFrame({
    n: pd.cut(runs[n]["jobs"].query("delay_tolerant").delay_s, bins, labels=labels)
         .value_counts().reindex(labels)
    for n in NAMES.values() if n != NAMES["on_arrival"]})
fig = vp.grouped_bars(delays, title="Start delay of the 1 456 delay-tolerant jobs",
                      y_title="Jobs (count)", x_title="Start delay")
fig.update_traces(texttemplate="%{y:,.0f}")
vp.save(fig, OUT / "start_delays", png=True)
fig.show()

### 3.4 Cost and peak

In [10]:
def cumulative_cost(r):
    ts = r["ts"]
    return ((ts.grid_kw.clip(lower=0) * ts.price / 1000) * r["step_s"] / 3600).cumsum() \
        .resample("15min").last()

c0 = cumulative_cost(runs[NAMES["on_arrival"]])
fig = vp.line_comparison(
    {n: cumulative_cost(runs[n]) - c0 for n in NAMES.values() if n != NAMES["on_arrival"]},
    title="Cumulative energy cost difference vs starting every job on arrival (within the day)",
    y_title="Cumulative cost difference (€)", shape="linear")
vp.save(fig, OUT / "cumulative_cost", png=True)
fig.show()

In [11]:
sel = kpis.T.loc[list(NAMES.values())]
fig = vp.tradeoff_scatter(sel, "Peak grid import, 15-min mean (kW)",
                          "Cost incl. job energy after midnight (€)",
                          title="Energy cost vs peak grid import per start rule (one day)",
                          x_title="Peak grid import, 15-min mean (kW)",
                          y_title="Energy cost incl. job energy after midnight (€)")
vp.save(fig, OUT / "cost_vs_peak", png=True)
fig.show()

### 3.5 Reactive hold through a Controller: threshold and the one-step delay

The `Controller` observes grid import *after* dispatch and can only change what the actor does at the next sample.
With a coarser step the hold decision lags more and released jobs start later.

In [12]:
cols = ["Cost incl. job energy after midnight (€)", "Peak grid import, 15-min mean (kW)", "Delay-tolerant jobs shifted",
        "Mean start delay, delay-tolerant (min)", "Max start delay, delay-tolerant (min)",
        "Deadline violations"]
sweep = pd.DataFrame({f"{t:,} kW": workload_kpis(r)[cols] for t, r in hold_sweep.items()}).T
sweep.index.name = "Import threshold"
display(vp.format_table(sweep, 1))
steps = pd.DataFrame({lbl: workload_kpis(r)[cols] for lbl, r in hold_steps.items()}).T
steps.index.name = "Step size"
vp.format_table(steps, 1)

,Cost incl. job energy after midnight (€),"Peak grid import, 15-min mean (kW)",Delay-tolerant jobs shifted,"Mean start delay, delay-tolerant (min)","Max start delay, delay-tolerant (min)",Deadline violations
Import threshold,,,,,,
"5,600 kW","10,800.6","6,324.1",218.0,6.8,119.0,0.0
"5,800 kW","10,812.3","6,386.3",182.0,4.0,85.0,0.0
"6,000 kW","10,817.0","6,435.3",145.0,2.6,70.0,0.0
"6,200 kW","10,819.1","6,480.3",88.0,1.4,55.0,0.0


,Cost incl. job energy after midnight (€),"Peak grid import, 15-min mean (kW)",Delay-tolerant jobs shifted,"Mean start delay, delay-tolerant (min)","Max start delay, delay-tolerant (min)",Deadline violations
Step size,,,,,,
1 min,"10,817.0","6,435.3",145.0,2.6,70.0,0.0
5 min,"10,816.6","6,480.3",131.0,2.3,70.0,0.0
15 min,"10,814.0","6,559.7",108.0,2.1,75.0,0.0


### 3.6 Custom code needed

In [13]:
pd.Series({
    "Server pool power model": logic_lines(ServerPool),
    "Cluster actor (admission, start rules, power)": logic_lines(ClusterActor),
    "Cooling proxy actor": logic_lines(CoolingProxy),
    "Reactive hold controller": logic_lines(ImportHoldController),
}, name="Lines of custom code (no comments or docstrings)").to_frame()

,Lines of custom code (no comments or docstrings)
Server pool power model,16
"Cluster actor (admission, start rules, power)",100
Cooling proxy actor,7
Reactive hold controller,6


## 4. Evaluation

- **H1 confirmed: Vessim has no workload model.** Demand is a power signal; jobs, queues, deadlines and CPU capacity
  had to be added by hand: ≈ 130 lines (server pool, cluster actor with start rules, cooling proxy, hold controller).
- **The retrofit is clean.** A custom `Actor` that overrides `sample()` is a documented extension point, and the
  `Recorder` logs the actor's extra state (waiting jobs, utilisation) without changes.
- **It reproduces the edge study.** The peak-aware greedy rule gives the edge run's IT power to within 10.9 kW on
  average (15-min means), its daily IT energy to 0.05 % and its peak IT power to 6 kW.
- **Workload flexibility is worth more than the battery here.** Shifting the 20 % delay-tolerant jobs lowers the
  15-min peak by ≈ 370–385 kW; the price rule saves ≈ 198 € per day (1.8 %) including the energy pushed past midnight.
  For comparison, the 500 kWh battery gave at most 250 kW and 77 € (E02).
- **Myopic greedy shifting delays almost every flexible job.** Only committed load is known, so later slots look
  emptier: 446 of 1 456 jobs start 2–3 h late under the peak rule. The edge scheduler behaves the same way.
- **Workarounds needed:**
  1. Vessim holds each actor sample for the whole step, so the actor returns the exact interval-mean power itself.
  2. `Controller`s act after dispatch: forecast-based rules live inside the actor; a reactive rule that needs the
     observed grid import must be a controller and lags one step (15-min step: peak +125 kW vs 1-min step).
  3. Actors cannot read each other: the cooling proxy reads the IT actor's attribute and relies on Vessim sampling
     actors in list order.
  4. Shifting moves energy past the horizon; costs must be corrected for it.
- **Speed.** One day with 7 279 jobs takes 0.7 s (start on arrival) to ≈ 5 s (forecast-based rules, pure-Python slot
  search). The peak rule reaches the edge result in ≈ 4 s, against ≈ 158 s for the edge study's scheduler, because
  each job is placed once on arrival instead of re-planning the whole queue every 30 s.